<a href="https://colab.research.google.com/github/Aashisrivatava/Customer_Churn_Data/blob/main/Customer_Churn_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

In [ ]:
df = pd.read_csv('https://github.com/ybifoundation/Dataset/raw/main/Bank%20Churn%20Modelling.csv')
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
df.info()
print(df.isna().sum())
print(df.Geography.value_counts())
print(df.Gender.value_counts())
print(df.Churn.value_counts(normalize=True))   # shows the ~80/20 imbalance

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerId        10000 non-null  int64  
 1   Surname           10000 non-null  object 
 2   CreditScore       10000 non-null  int64  
 3   Geography         10000 non-null  object 
 4   Gender            10000 non-null  object 
 5   Age               10000 non-null  int64  
 6   Tenure            10000 non-null  int64  
 7   Balance           10000 non-null  float64
 8   Num Of Products   10000 non-null  int64  
 9   Has Credit Card   10000 non-null  int64  
 10  Is Active Member  10000 non-null  int64  
 11  Estimated Salary  10000 non-null  float64
 12  Churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 1015.8+ KB
CustomerId          0
Surname             0
CreditScore         0
Geography           0
Gender              0
Age         

In [ ]:
y = df['Churn']
X = df[['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
        'Num Of Products', 'Has Credit Card', 'Is Active Member', 'Estimated Salary']]

X = pd.get_dummies(X, drop_first=True).astype('float')
X.shape

(10000, 11)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2529, stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_test = ss.transform(X_test)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

w = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: w[0], 1: w[1]}
class_weight

{0: np.float64(0.6279434850863422), 1: np.float64(2.4539877300613497)}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[tf.keras.metrics.AUC(name='auc')])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 929 (3.63 KB)

 Trainable params: 929 (3.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_auc', mode='max', patience=8,
                   restore_best_weights=True)

history = model.fit(X_train, y_train,
                    epochs=100, batch_size=32,
                    validation_split=0.2,
                    class_weight=class_weight,
                    callbacks=[es], verbose=1)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - auc: 0.6748 - loss: 0.6408 - val_auc: 0.7752 - val_loss: 0.5529
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.7799 - loss: 0.5657 - val_auc: 0.8036 - val_loss: 0.5276
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8054 - loss: 0.5376 - val_auc: 0.8239 - val_loss: 0.5035
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8243 - loss: 0.5148 - val_auc: 0.8333 - val_loss: 0.4798
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8303 - loss: 0.5053 - val_auc: 0.8372 - val_loss: 0.4687
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8373 - loss: 0.4953 - val_auc: 0.8392 - val_loss: 0.4565
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8424 - loss: 0.4899 - val_auc: 0.8413 - val_loss: 0.4538
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.8405 - loss: 0.4908 - val_auc: 0.8428 - val_loss: 0.4516
Epoch 9/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/

In [ ]:
from sklearn.metrics import roc_auc_score

proba = model.predict(X_test).ravel()
print("ROC-AUC:", roc_auc_score(y_test, proba))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
ROC-AUC: 0.8477553053824239


In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
best_thr = thr[np.argmax(f1[:-1])]
print("Best threshold:", best_thr)

Best threshold: 0.64008045


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (proba >= best_thr).astype(int)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.91      0.91      1593
           1       0.64      0.59      0.61       407

    accuracy                           0.85      2000
   macro avg       0.77      0.75      0.76      2000
weighted avg       0.84      0.85      0.85      2000

[[1456  137]
 [ 168  239]]


In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                    eval_metric='auc', random_state=42)
xgb.fit(X_train, y_train)

p = xgb.predict_proba(X_test)[:, 1]
print("XGB ROC-AUC:", roc_auc_score(y_test, p))
print(classification_report(y_test, (p >= 0.5).astype(int)))

XGB ROC-AUC: 0.8613898952882003
              precision    recall  f1-score   support

           0       0.92      0.81      0.86      1593
           1       0.50      0.73      0.59       407

    accuracy                           0.79      2000
   macro avg       0.71      0.77      0.73      2000
weighted avg       0.84      0.79      0.81      2000

